# 18. Reproducibilidad y bibliografía

Este último capítulo reúne lo necesario para reconstruir los resultados del libro. La guía (sección 9.2) pide una lista de verificación que cubra:
- las versiones del software y el hardware;
- las semillas;
- los rangos de hiperparámetros;
- el número de ejecuciones y las medidas de dispersión;
- los enlaces a los datos y al código;
- el tiempo de cómputo.

Cada sección cubre uno de esos puntos y lee sus datos del repositorio, no de valores escritos a mano. Al final están la lista de verificación y la bibliografía.

## 18.1. Entorno de software y hardware

Todos los experimentos se ejecutaron en el equipo de los autores, con el entorno conda `ml_proyecto` (Python 3.12.13). `requirements.txt` fija la versión exacta de cada librería, y el dashboard en producción usa el subconjunto de `requirements-render.txt`.

La celda siguiente registra el equipo en que se ejecuta el libro. Además, compara las librerías principales con las versiones fijadas: si alguien reproduce el libro con otras versiones, la última columna lo señala.

In [1]:
from config import *
from IPython.display import display
import os
import platform
import re
from importlib import metadata

import psutil

equipo = pd.Series({
    "sistema operativo": f"{platform.system()} {platform.release()}",
    "procesador": platform.processor() or platform.machine(),
    "núcleos físicos / lógicos": f"{psutil.cpu_count(logical=False)} / {os.cpu_count()}",
    "memoria RAM (GB)": round(psutil.virtual_memory().total / 1024**3, 1),
    "GPU": "no se usó",
    "Python": platform.python_version(),
}, name="valor")
display(equipo.to_frame())

fijadas = dict(re.findall(r"^([A-Za-z0-9_.\-]+)==([^\s;]+)",
                          (RAIZ / "requirements.txt").read_text(encoding="utf-8"), flags=re.M))
fijadas = {k.lower(): v for k, v in fijadas.items()}
principales = ["numpy", "pandas", "scipy", "scikit-learn", "xgboost", "lightgbm", "catboost",
               "imbalanced-learn", "optuna", "deap", "statsmodels", "joblib", "matplotlib",
               "seaborn", "dash", "plotly", "pytest", "jupyter-book"]
def _version(paquete):
    try:
        return metadata.version(paquete)
    except metadata.PackageNotFoundError:
        return "no instalada"

versiones = pd.DataFrame([{"librería": p, "instalada": _version(p),
                           "fijada en requirements.txt": fijadas.get(p, "—")} for p in principales])
versiones["coincide"] = versiones["instalada"] == versiones["fijada en requirements.txt"]
display(versiones)

,valor
sistema operativo,Windows 11
procesador,"AMD64 Family 23 Model 160 Stepping 0, Authenti..."
núcleos físicos / lógicos,4 / 8
memoria RAM (GB),15.3
GPU,no se usó
Python,3.12.13


,librería,instalada,fijada en requirements.txt,coincide
0,numpy,2.5.3,2.5.3,True
1,pandas,3.0.5,3.0.5,True
2,scipy,1.18.1,1.18.1,True
3,scikit-learn,1.9.1,1.9.1,True
4,xgboost,3.4.1,3.4.1,True
5,lightgbm,4.7.0,4.7.0,True
6,catboost,1.2.10,1.2.10,True
7,imbalanced-learn,0.14.2,0.14.2,True
8,optuna,5.0.0,5.0.0,True
9,deap,1.4.4,1.4.4,True


## 18.2. Semillas

Toda fuente de azar parte de una semilla fija. `src/configuracion.py` define `SEMILLA = 42` y la propaga a numpy, scikit-learn, XGBoost, LightGBM, CatBoost, Optuna y DEAP.

| Dónde se usa | Semilla | Capítulo |
|---|---|---|
| Partición entrenamiento/prueba por paciente (80/20) | 42 | 2 |
| Cinco folds externos `StratifiedGroupKFold` agrupados por paciente | 42 | 8, 12, 13 |
| Tres folds internos de la búsqueda de hiperparámetros | 42 | 8, 12 |
| Muestreo de la búsqueda (Random, Optuna, DEAP) y submuestra del 25 % de pacientes | 42 | 8, 12 |
| Conjunto de selección de DES-CPL (tres pliegues `StratifiedGroupKFold`) | 42 | 13 |
| Búsqueda de DES-CPL con Optuna | 42 + número de fold | 13 |
| Conjuntos adicionales: folds, submuestra de CDC y barajado de Bank Marketing | 42 | 16 |
| Estudio de varianza por semilla | 42, 7, 2026, 1, 2, 3, 4, 5, 6 y 8 | 16 |
| Variación por semilla del Entregable 2 | 42, 7 y 2026 | 11 |

La comparación confirmatoria usa solo la semilla 42, fijada en el protocolo antes de ver los resultados. Las demás son análisis de varianza.

## 18.3. Rangos de hiperparámetros

La celda siguiente imprime los espacios de búsqueda directamente desde el código que los usa, para que la tabla no pueda desactualizarse:
- `src/espacios.py`, para los siete modelos base;
- `src/sota.py`, para LightGBM y CatBoost;
- `src/experimento_des_cpl.py`, para DES-CPL.

| Notación | Significado |
|---|---|
| `log` | Real con distribución log-uniforme |
| `float` | Real con distribución uniforme |
| `int` | Entero con distribución uniforme |
| `intlog` | Entero con distribución log-uniforme |
| `cat` | Categórica |

El Entregable 2 usó además una rejilla gruesa de 8 a 12 puntos para Grid Search. Desde el capítulo 12, todos los modelos se comparan con 60 evaluaciones.

In [2]:
from src import espacios, sota
from src import experimento_des_cpl as exd

def _formato(dist):
    tipo, *valores = dist
    return f"{tipo} {valores[0]}" if tipo == "cat" else f"{tipo} [{valores[0]:g}, {valores[1]:g}]"

filas = []
for modelo in ["logistica", "bayes", "knn", "arbol", "random_forest", "xgboost", "svm"]:
    for parametro, dist in espacios.espacio_busqueda(modelo).items():
        filas.append({"modelo": modelo, "hiperparámetro": parametro.split("__")[-1], "espacio": _formato(dist)})
for modelo, espacio in sota.ESPACIOS.items():
    for parametro, dist in espacio.items():
        filas.append({"modelo": modelo, "hiperparámetro": parametro.split("__")[-1], "espacio": _formato(dist)})
for nombre, valores in [("k", exd.K_REJILLA), ("tau", exd.TAU_REJILLA),
                        ("lambda (encogimiento)", exd.LAMBDA_REJILLA),
                        ("modelos tras la poda", ["4" if v else "7 (sin poda)" for v in exd.PODA_REJILLA])]:
    filas.append({"modelo": "DES-CPL", "hiperparámetro": nombre, "espacio": f"cat {list(valores)}"})
hiperparametros = pd.DataFrame(filas)
display(hiperparametros)
print(f"Evaluaciones por fold externo: 60 para todos los modelos (exd.PRESUPUESTO = {exd.PRESUPUESTO})")

,modelo,hiperparámetro,espacio
0,logistica,C,"log [0.0001, 100]"
1,logistica,penalty,"cat ['l1', 'l2']"
2,bayes,var_smoothing,"log [1e-12, 1]"
3,knn,n_neighbors,"intlog [5, 301]"
4,knn,weights,"cat ['uniform', 'distance']"
5,knn,p,"cat [1, 2]"
6,arbol,max_depth,"int [2, 30]"
7,arbol,min_samples_leaf,"intlog [1, 500]"
8,arbol,criterion,"cat ['gini', 'entropy']"
9,arbol,ccp_alpha,"log [1e-07, 0.01]"


Evaluaciones por fold externo: 60 para todos los modelos (exd.PRESUPUESTO = 60)


## 18.4. Tabla maestra de experimentos

`results/tabla_maestra_unificada.csv`, que genera `experiments/tabla_maestra_unificada.py`, reúne en un solo archivo todo lo evaluado en el bucle externo de los dos entregables:
- las 112 combinaciones del Entregable 2;
- los modelos con presupuesto de 60 evaluaciones;
- el estado del arte, los ensambles estáticos, DES-CPL y sus ablaciones;
- los conjuntos adicionales, las semillas y los escenarios de robustez.

Cada fila es un método en un conjunto de datos, con los datos de sus 5 folds externos: la media y la desviación del AUC-PR, el valor de cada fold, el presupuesto, la semilla y, cuando se registró, el tiempo. La guía hablaba de 140 combinaciones en el Entregable 2. Este proyecto tiene 112, porque los autores decidieron limitarlo a la rama de clasificación.

In [3]:
maestra = pd.read_csv(RESULTADOS / "tabla_maestra_unificada.csv")
resumen = (maestra.groupby(["bloque", "conjunto"], sort=False)
           .agg(filas=("método", "size"), folds=("folds", "max"),
                evaluaciones=("evaluaciones", lambda s: "/".join(str(int(v)) for v in sorted(s.dropna().unique()))),
                semillas=("semilla", lambda s: s.dropna().nunique()),
                auc_pr_sd_mediana=("auc_pr_sd", "median")))
display(resumen.round(4))
print(f"{len(maestra)} filas; {int(maestra['folds'].fillna(0).sum())} evaluaciones de fold externo en total")

filas  folds evaluaciones  semillas  auc_pr_sd_mediana
bloque                                        conjunto                                                                         
E2: experimento combinatorio                  diabetes                     112    5.0         8/12         1             0.0121
E3: modelos base con presupuesto 60 (cap. 12) diabetes                       7    5.0           60         1             0.0124
E3: ensambles estáticos                       diabetes                       3    5.0           60         1             0.0158
E3: modelo original                           diabetes                       1    5.0           60         1             0.0174
E3: modelo original (versión inicial)         diabetes                       1    5.0           60         1             0.0131
E3: ablación                                  diabetes                       7    5.0           60         1             0.0170
E3: estado del arte                           diabetes                       2    5.0           60         1             0.0157
E3: conjuntos adicionales                     credito                       13    5.0           60         1             0.0051
                                              banco                         13    5.0           60         1             0.0166
                                              cdc                           13    5.0           60         1             0.0044
E3: varianza por semilla                      diabetes                      20    5.0           60        10             0.0153
E3: robustez                                  diabetes (ruido_05)           11    5.0           60         1             0.0137
                                              diabetes (ruido_10)           11    5.0           60         1             0.0140
                                              diabetes (tamano_50)          11    5.0           60         1             0.0122
                                              diabetes (prevalencia_05)     11    5.0           60         1             0.0142
                                              diabetes (tamano_25)          11    5.0           60         1             0.0117

247 filas; 1215 evaluaciones de fold externo en total


**Número de ejecuciones y dispersión.**
- Cada fila se evaluó en los **5 folds externos**, con una sola repetición del bucle; el protocolo declara la desviación respecto al 5 × 5 que recomienda la guía.
- Como medida de dispersión se reportan la **desviación estándar entre folds** (columna `auc_pr_sd`) y, en el conjunto de prueba, los **intervalos BCa** remuestreando pacientes (capítulos 11 y 13).
- La **varianza por semilla** se midió con diez semillas para DES-CPL y XGBoost (capítulo 16).

## 18.5. Tiempo de cómputo

La tabla suma los tiempos que cada experimento registró en sus archivos de `results/`.

Dos advertencias para leerla:
- **Tiempo de trabajo, no de reloj.** Los tiempos se midieron por ajuste. Con 4 procesos en paralelo, el tiempo de reloj puede ser menor.
- **Cota inferior.** No incluyen la búsqueda propia de DES-CPL, el cálculo de métricas ni las pruebas estadísticas. La sección 16.8 detalla el costo por modelo.

Las carpetas `results/robustez/` y `results/semillas/` no se versionan, porque son intermedios grandes. En un clon nuevo esas filas aparecen sin dato hasta que se vuelven a correr los experimentos.

In [4]:
import glob

def _horas(patrones, columnas):
    patrones = [patrones] if isinstance(patrones, str) else patrones
    archivos = [a for p in patrones for a in glob.glob(str(RESULTADOS / p))]
    if not archivos:
        return np.nan
    tablas = [pd.read_csv(a) for a in archivos]
    return sum(t[[c for c in columnas if c in t]].fillna(0).to_numpy().sum() for t in tablas) / 3600

etapas = [
    ("E2: 112 combinaciones (cap. 8)", "tabla_maestra_clasificacion.csv", ["tiempo_total_s"]),
    ("E3: convergencia, Random 60 (cap. 12)", "convergencia_evaluaciones.csv", ["tiempo_s"]),
    ("E3: convergencia, Optuna 60 (cap. 12)", "convergencia_optuna.csv", ["tiempo_total_s"]),
    ("E3: LightGBM y CatBoost (cap. 13)", "benchmarks_sota.csv", ["tiempo_busqueda_s", "tiempo_ajuste_s", "tiempo_inferencia_s"]),
    ("E3: modelos base de DES-CPL, 5 folds (cap. 13)", "des_cpl_60/tiempos.csv", ["t_pliegues_s", "t_final_s"]),
    ("E3: evaluación en prueba (cap. 13.8)", "des_cpl_prueba/tiempos.csv", ["t_pliegues_s", "t_final_s"]),
    ("E3: conjuntos adicionales, búsqueda (cap. 16)", [f"adicionales/{c}/tiempos_busqueda.csv" for c in ["credito", "banco", "cdc"]], ["tiempo_s"]),
    ("E3: conjuntos adicionales, ajustes (cap. 16)", [f"adicionales/{c}/tiempos.csv" for c in ["credito", "banco", "cdc"]], ["t_pliegues_s", "t_final_s"]),
    ("E3: robustez, 25 folds (cap. 16)", "robustez/*/tiempos.csv", ["t_pliegues_s", "t_final_s"]),
    ("E3: semillas, 45 folds (cap. 16)", [f"semillas/s{s}/tiempos.csv" for s in [7, 2026, 1, 2, 3, 4, 5, 6, 8]], ["t_pliegues_s", "t_final_s"]),
]
computo = pd.DataFrame([{"etapa": e, "horas": _horas(p, c)} for e, p, c in etapas]).set_index("etapa")
display(computo.round(2))
print(f"Total registrado: {computo['horas'].sum():.1f} horas de trabajo")

,horas
etapa,
E2: 112 combinaciones (cap. 8),22.91
"E3: convergencia, Random 60 (cap. 12)",5.13
"E3: convergencia, Optuna 60 (cap. 12)",1.95
E3: LightGBM y CatBoost (cap. 13),1.92
"E3: modelos base de DES-CPL, 5 folds (cap. 13)",0.44
E3: evaluación en prueba (cap. 13.8),0.12
"E3: conjuntos adicionales, búsqueda (cap. 16)",4.83
"E3: conjuntos adicionales, ajustes (cap. 16)",0.44
"E3: robustez, 25 folds (cap. 16)",1.89


Total registrado: 44.4 horas de trabajo


El grueso del cómputo se fue en el experimento combinatorio del Entregable 2 (unas 23 horas). La comparación del Entregable 3 reutiliza las predicciones fuera de pliegue de cada modelo base. Por eso todas las variantes de DES-CPL, las ablaciones y los ensambles estáticos se evalúan sin reentrenar nada.

## 18.6. Cómo reproducir los resultados

Desde la raíz del repositorio, con el entorno activado. En PowerShell 5, los comandos van en líneas separadas, porque `&&` no funciona.

```bash
conda create -n ml_proyecto python=3.12.13
conda activate ml_proyecto
pip install -r requirements.txt

pytest -m "not lento"            # pruebas rápidas (las mismas de GitHub Actions)
pytest -m lento                  # reproduce una corrida de la tabla maestra (~7 min)

# Entregable 2: ejecutar los notebooks 01 a 11 desde docs/ (el 08 corre el experimento, ~23 h).
# Entregable 3, en este orden:
python experiments/benchmarks_sota.py          # LightGBM y CatBoost
python experiments/des_cpl.py --folds 0 1 2 3 4   # DES-CPL, presupuesto 60
python experiments/evaluacion_prueba.py        # conjunto de prueba (cap. 13.8)
python experiments/conjuntos_adicionales.py    # crédito, Bank Marketing y CDC
python experiments/semillas_robustez.py        # sensibilidad, robustez y 10 semillas
python experiments/entrenar_artefactos.py      # artefactos de los 7 modelos
python experiments/artefacto_des_cpl.py        # artefacto de DES-CPL, SHAP y tiempos
python experiments/tabla_maestra_unificada.py  # tabla maestra
python experiments/revision_literatura.py      # búsqueda bibliográfica (cap. 14)

# El capítulo 12 calcula su estudio de convergencia al ejecutarse.
cd docs
jupyter-book build .                          # el libro no reejecuta los notebooks
python ../app.py                               # dashboard en http://127.0.0.1:8050
```

**Reanudación.** Cada script reanuda donde quedó: guarda un punto de control por fold o por lote y, al volver a correrlo, solo calcula lo que falta. Para rehacer una parte, basta con borrar su archivo de `results/`.

**Exactitud de la reproducción.** Seis de los siete modelos base reproducen el Entregable 2 sin diferencias. Bayes difiere hasta en 0.0012 por fold (sección 13.3.1). La prueba lenta reproduce fold a fold una corrida de la tabla maestra.

## 18.7. Lista de verificación de reproducibilidad

| Elemento | Estado | Dónde |
|---|---|---|
| Versiones del software | ✔ | `requirements.txt` (versiones exactas), sección 18.1 |
| Hardware | ✔ | Sección 18.1 (4 núcleos físicos, `N_JOBS = 4`, sin GPU) |
| Semillas | ✔ | Sección 18.2, `src/configuracion.py` |
| Rangos de hiperparámetros | ✔ | Sección 18.3, `src/espacios.py`, `src/sota.py`, `src/experimento_des_cpl.py` |
| Presupuesto de ajuste por modelo | ✔ | 60 evaluaciones por fold externo; sección 16.8 y `results/presupuesto_tiempos.csv` |
| Número de ejecuciones | ✔ | 5 folds externos × 1 repetición; 10 semillas en el estudio de varianza (sección 18.4) |
| Medidas de dispersión | ✔ | Desviación entre folds, BCa por paciente, varianza por semilla |
| Protocolo pre-registrado | ✔ | `experiments/protocolo.md`, etiqueta `protocolo-v1` (9 de noviembre de 2026) |
| Tabla maestra de experimentos | ✔ | `results/tabla_maestra_unificada.csv` (sección 18.4) |
| Modelos serializados | ✔ | `artifacts/*.joblib` con `metadata.json` y prueba de ida y vuelta |
| Pruebas automáticas | ✔ | `tests/` (156 pruebas), GitHub Actions (`.github/workflows/pruebas.yml`) |
| Ficha del modelo y declaración de datos | ✔ | `MODEL_CARD.md`, `DATA_STATEMENT.md` |
| Enlaces a los datos | ✔ | Sección 18.8 |
| Enlace al código | ✔ | Sección 18.8 |
| Tiempo de cómputo | ✔ | Sección 18.5 |
| Archivo con identificador persistente (DOI) | **Pendiente** | Se crea en Zenodo desde la versión final (sección 18.8) |
| URL pública del dashboard | **Pendiente** | Despliegue en Render (sección 18.8) |

## 18.8. Datos, código y archivo

**Datos.**

| Conjunto | Fuente | Uso |
|---|---|---|
| Diabetes 130-US Hospitals for Years 1999-2008 | UCI, DOI [10.24432/C5230J](https://doi.org/10.24432/C5230J); copia en `data/raw/diabetic_data.csv` | Problema principal |
| Default of Credit Card Clients | [UCI, conjunto 350](https://archive.ics.uci.edu/dataset/350) | Conjunto adicional |
| Bank Marketing | [UCI, conjunto 222](https://archive.ics.uci.edu/dataset/222) | Conjunto adicional |
| CDC Diabetes Health Indicators | [UCI, conjunto 891](https://archive.ics.uci.edu/dataset/891) | Conjunto adicional |

Los conjuntos adicionales los descarga `experiments/conjuntos_adicionales.py` en `data/raw/adicionales/`. Esa carpeta no se versiona. La declaración completa de los datos está en `DATA_STATEMENT.md`.

**Código y productos.**
- **Repositorio:** <https://github.com/joshua2412-a/Diabetes_ProyectoML_202630>, licencia MIT.
- **Libro:** <https://joshua2412-a.github.io/Diabetes_ProyectoML_202630>.
- **Dashboard:** *[URL de Render pendiente del despliegue]*.

**Archivo versionado.** *[DOI de Zenodo pendiente: 10.5281/zenodo.XXXXXXX]*. Se genera desde la versión (*release*) de GitHub que acompaña la entrega. Para citar el proyecto, usar `CITATION.cff`.

## 18.9. Bibliografía

Referencias citadas en el libro, en formato BibTeX en `docs/references.bib`. Los trabajos encontrados en la revisión de literatura, con sus enlaces, están además en las tablas del capítulo 14 y en `results/literatura/`.

```{bibliography}
:all:
```